
[![Electronic Profile](https://img.shields.io/badge/Electronic%20Profile-engineer--e-181717?logo=github)](https://github.com/engineer-e/) 
[![Work Profile](https://img.shields.io/badge/Work%20Profile-engineer--work-181717?logo=github)](https://github.com/engineer-work/) 
[![Instagram](https://img.shields.io/badge/Instagram-gobalkrishnan.engineer-E4405F?logo=instagram&logoColor=white)](https://www.instagram.com/gobalkrishnan.engineer/)

# 6. Surface Normals and Multiple Objects 

## 6.3 An Abstraction for Hittable Objects   

In [5]:
import sys 
sys.path.append("../../Ray Tracing in One Weekend/")

from util.ray import Ray
from util.vec3 import Vec3
from util.color import write_color
from util.vec3 import unit_vector
from util.vec3 import dot
from tqdm import tqdm
import math


# Hit Record


In [6]:
class HitRecord:
    def __init__(self):
        self.p = None
        self.normal = None
        self.t = 0.0

# Hittable (Abstract Class)

In [7]:
from abc import ABC, abstractmethod

class Hittable(ABC):

    @abstractmethod
    def hit(self, ray, tmin, tmax, rec):
        pass

# Sphere (Core conversion)

In [9]:
class Sphere(Hittable):
    def __init__(self, center, radius):
        self.center = center
        self.radius = max(0.0, radius)

    def hit(self, ray, tmin, tmax, rec):

        oc = self.center - ray.origin

        a = ray.direction.length_squared()
        h = ray.direction.dot(oc)
        c = oc.length_squared() - self.radius * self.radius

        discriminant = h*h - a*c

        if discriminant < 0:
            return False

        sqrtd = math.sqrt(discriminant)

        # first root
        root = (h - sqrtd) / a
        if root <= tmin or root >= tmax:
            root = (h + sqrtd) / a
            if root <= tmin or root >= tmax:
                return False

        rec.t = root
        rec.p = ray.at(rec.t)

        # normal = (P - center) / radius
        rec.normal = (rec.p - self.center) * (1.0 / self.radius)

        return True



# 1. Big Idea (Why “hittable” exists)

Instead of:

* list of spheres ❌ (limited design)

We define:

> “Anything a ray can hit” = **hittable**

So:

* Sphere → hittable
* Future fog / plane / box → also hittable
* Even list of objects → hittable

This is **polymorphism idea** (same interface, many shapes).

---

# 2. Core Math Behind Sphere Hit

## Ray equation

$$
P(t) = O + tD
$$

| Symbol | Meaning            |
| ------ | ------------------ |
| $O$    | ray origin         |
| $D$    | ray direction      |
| $t$    | distance along ray |
| $P(t)$ | point on ray       |

---

## Sphere equation

$$
|P - C|^2 = r^2
$$

| Symbol | Meaning       |
| ------ | ------------- |
| $C$    | sphere center |
| $r$    | radius        |

---

## Combine them → quadratic

Substitute ray into sphere:

$$
|O + tD - C|^2 = r^2
$$

This becomes:

$$
at^2 - 2ht + c = 0
$$

where:

| Term | Formula           | Meaning               |         
| ---- | ----------------- | --------------------- | 
| $a$  | $D \cdot D$       | direction energy      |          
| $h$  | $D \cdot (C - O)$ | alignment with center |          
| $c$  | $C - O^2 - r^2$   | offset from sphere    |

---

## Discriminant

$$
\Delta = h^2 - ac
$$

| Case         | Meaning |
| ------------ | ------- |
| $\Delta < 0$ | no hit  |
| $\Delta = 0$ | tangent |
| $\Delta > 0$ | 2 hits  |

---

## Roots

$$
t = \frac{h \pm \sqrt{\Delta}}{a}
$$

We choose:

* **smallest valid t first (nearest hit)**

---

# 3. hit_record (VERY IMPORTANT IDEA)

| Field  | Meaning            |
| ------ | ------------------ |
| p      | hit point          |
| normal | surface direction  |
| t      | distance along ray |

👉 This is “what we store when something is hit”.

---

# 4. Python Translation (Clean Version)

```python
import math
from abc import ABC, abstractmethod
```

---

## Vector helpers (minimal)

```python
class Vec3:
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

    def __add__(self, other):
        return Vec3(self.x + other.x, self.y + other.y, self.z + other.z)

    def __sub__(self, other):
        return Vec3(self.x - other.x, self.y - other.y, self.z - other.z)

    def __mul__(self, t):
        return Vec3(self.x * t, self.y * t, self.z * t)

    def dot(self, other):
        return self.x * other.x + self.y * other.y + self.z * other.z

    def length_squared(self):
        return self.dot(self)
```

---

## Ray

```python
class Ray:
    def __init__(self, origin, direction):
        self.origin = origin
        self.direction = direction

    def at(self, t):
        return self.origin + self.direction * t
```

---

## Hit Record

```python
class HitRecord:
    def __init__(self):
        self.p = None
        self.normal = None
        self.t = 0.0
```

---

## Hittable (Abstract Class)

```python
class Hittable(ABC):

    @abstractmethod
    def hit(self, ray, tmin, tmax, rec):
        pass
```

---

## Sphere (Core conversion)

```python
class Sphere(Hittable):
    def __init__(self, center, radius):
        self.center = center
        self.radius = max(0.0, radius)

    def hit(self, ray, tmin, tmax, rec):

        oc = self.center - ray.origin

        a = ray.direction.length_squared()
        h = ray.direction.dot(oc)
        c = oc.length_squared() - self.radius * self.radius

        discriminant = h*h - a*c

        if discriminant < 0:
            return False

        sqrtd = math.sqrt(discriminant)

        # first root
        root = (h - sqrtd) / a
        if root <= tmin or root >= tmax:
            root = (h + sqrtd) / a
            if root <= tmin or root >= tmax:
                return False

        rec.t = root
        rec.p = ray.at(rec.t)

        # normal = (P - center) / radius
        rec.normal = (rec.p - self.center) * (1.0 / self.radius)

        return True
```

---

# 5. Key Learning Table (VERY IMPORTANT)

## A. Why quadratic appears

| Step            | Meaning                |
| --------------- | ---------------------- |
| Ray + sphere    | intersection condition |
| expand equation | algebra                |
| quadratic form  | 2 possible hits        |

---

## B. Why use `h` instead of `b`

| Version   | Formula                    |
| --------- | -------------------------- |
| classic   | (t = (-b ± √D) / 2a)       |
| optimized | (t = (h ± √(h² - ac)) / a) |

👉 reduces multiplication and simplifies code

---

## C. Why two roots?

| Root      | Meaning           |
| --------- | ----------------- |
| smaller t | first surface hit |
| larger t  | exit point        |

We always pick:

> smallest valid positive t

---

## D. Why interval [tmin, tmax]?

| Purpose        | Reason                      |
| -------------- | --------------------------- |
| avoid self-hit | ignore very small t         |
| limit search   | only closest object matters |
| ray bouncing   | prevents artifacts          |

---

## E. Normal vector meaning

$$
normal = \frac{P - C}{r}
$$

| Meaning   | Explanation         |
| --------- | ------------------- |
| direction | outward from center |
| length    | normalized          |
| purpose   | lighting direction  |

---

# 6. Concept Summary (Simple)

* Ray = path of light
* Sphere = math equation
* Intersection = solving quadratic
* hittable = “anything ray can touch”
* hit_record = “what we remember after hit”

---

If you want next step, I can show:

👉 how to make **multiple spheres (HittableList)**
👉 or how this becomes a **full ray tracer scene system**
👉 or visualize the math step-by-step in geometry diagrams
